# 3. Enrich text

Async fan-out over `posts.parquet` (from notebook 2) through the text resolver. Bluesky and Reddit posts get full-text enrichment; every other platform falls back to `posts.title`.

Inputs: `output/<RUN_ID>/posts.parquet`.
Outputs: `output/<RUN_ID>/resolved_posts.parquet` (one row per input post).

See `PIPELINE_PLAN.md` § S20 / S6 / S7 / S8.

## Setup

Load `.env` (for Reddit creds), set `RUN_ID` to the latest run that has a `posts.parquet`.

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv

REPO_ROOT = Path.cwd().parent.resolve()
load_dotenv(REPO_ROOT / ".env")

OUTPUT_ROOT = REPO_ROOT / "altendor" / "output"

# Override RUN_ID here if you want a specific historical run instead of the latest.
RUN_ID: str | None = None
if RUN_ID is None:
    candidates = sorted([p.name for p in OUTPUT_ROOT.iterdir() if (p / "posts.parquet").exists()])
    if not candidates:
        raise FileNotFoundError("No run with posts.parquet found. Run notebook 2 first.")
    RUN_ID = candidates[-1]

OUT_DIR = OUTPUT_ROOT / RUN_ID
print(f"Using run id: {RUN_ID}\nOutput dir: {OUT_DIR}")

Using run id: 2026-06-28T21-34
Output dir: /home/automathis/Documents/Codebases/ICSSI-2026-Hackaton/altendor/output/2026-06-28T21-34


In [2]:
import pandas as pd

posts_df = pd.read_parquet(OUT_DIR / "posts.parquet")
print(f"Loaded {len(posts_df)} posts")

Loaded 17218 posts


## Async resolver

Bluesky resolution is unauthenticated (public AppView). Reddit requires `REDDIT_CLIENT_ID`, `REDDIT_CLIENT_SECRET`, and `REDDIT_USER_AGENT` in the environment; if any are missing, Reddit rows fall back to `posts.title`.

In [3]:
import asyncio

import aiohttp
import nest_asyncio

# Patch the running Jupyter event loop so `asyncio.run` works inside cells.
nest_asyncio.apply()

from altendor.enrich.reddit import get_reddit_client
from altendor.enrich.text_resolver import resolve_full_text

reddit_client = get_reddit_client()
if reddit_client is None:
    print("Reddit creds missing — Reddit posts will fall back to posts.title.")
else:
    print("Reddit client built (read-only).")

Reddit creds missing — Reddit posts will fall back to posts.title.


## Run resolver

One coroutine per row, gathered concurrently. The bluesky module owns its own semaphore; reddit calls are sequential (PRAW is sync). Exceptions are counted, not raised.

In [4]:
async def resolve_all(posts_df, reddit_client):
    async with aiohttp.ClientSession() as bsky_session:
        coros = [
            resolve_full_text(row.to_dict(), bsky_session=bsky_session, reddit_client=reddit_client)
            for _, row in posts_df.iterrows()
        ]
        # Honour the bluesky module's semaphore + sequential reddit; just await them all.
        return await asyncio.gather(*coros, return_exceptions=True)


# Works in any Jupyter setup thanks to nest_asyncio.apply() above (handles
# Jupyter's already-running event loop). Use this pattern in any notebook
# that needs to drive async code.
resolved_raw = asyncio.run(resolve_all(posts_df, reddit_client))

resolved_rows = []
n_errors = 0
for r in resolved_raw:
    if isinstance(r, Exception):
        n_errors += 1
        continue
    resolved_rows.append(r)

print(f"Resolved {len(resolved_rows)} / {len(posts_df)}; errors: {n_errors}")

Resolved 17218 / 17218; errors: 0


## Convert to DataFrame

In [5]:
from dataclasses import asdict

resolved_df = pd.DataFrame([asdict(r) for r in resolved_rows])
print(f"Resolved DataFrame: {len(resolved_df)} rows")
resolved_df["platform"].value_counts()

Resolved DataFrame: 17218 rows


platform
twitter        12593
news            1714
reddit           683
blog             609
policy           602
other            510
wikipedia        434
podcast           52
video             14
patent             5
peer_review        2
Name: count, dtype: int64

In [6]:
print("text_confidence breakdown:")
print(resolved_df["text_confidence"].value_counts().to_string())
print("\nlow-confidence rows per platform:")
print(resolved_df[resolved_df.text_confidence == "low"]["platform"].value_counts().to_string())

text_confidence breakdown:
text_confidence
low     17135
high       83

low-confidence rows per platform:
platform
twitter        12593
news            1698
reddit           675
blog             593
policy           562
other            508
wikipedia        434
podcast           51
video             14
patent             5
peer_review        2


## Write outputs

Persist `resolved_posts.parquet` and append to the run's `manifest.json`.

In [7]:
import json

out_path = OUT_DIR / "resolved_posts.parquet"
resolved_df.to_parquet(out_path, index=False)

manifest_path = OUT_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {}
manifest["stage_3_enrich_text"] = {
    "n_input": int(len(posts_df)),
    "n_resolved": int(len(resolved_df)),
    "n_errors": int(n_errors),
    "by_platform": {str(k): int(v) for k, v in resolved_df["platform"].value_counts().items()},
    "low_confidence_count": int((resolved_df["text_confidence"] == "low").sum()),
}
manifest.setdefault("output_files", []).append(str(out_path.relative_to(REPO_ROOT)))
manifest_path.write_text(json.dumps(manifest, indent=2, sort_keys=True))
print(f"Wrote {out_path}")

Wrote /home/automathis/Documents/Codebases/ICSSI-2026-Hackaton/altendor/output/2026-06-28T21-34/resolved_posts.parquet
